# 06 — Activity & Transactions

Explore league activity: recent moves, trades, waiver claims, and the transaction API.

In [ ]:
import sys
sys.path.insert(0, "../src")
from connection import get_league
import pandas as pd
from datetime import datetime

league = get_league()
print(f"Connected: {league.settings.name}")

## Recent Activity

`league.recent_activity()` returns a list of `Activity` objects.
Each has:
- `actions`: list of `(team, action_type, player_name, position)` tuples
- `date`: timestamp

In [ ]:
activities = league.recent_activity(size=50)
print(f"Retrieved {len(activities)} activities\n")

rows = []
for act in activities:
    date = datetime.fromtimestamp(act.date / 1000).strftime('%Y-%m-%d %H:%M')
    for team, action, player, position in act.actions:
        team_name = team.team_name if hasattr(team, 'team_name') else str(team)
        rows.append({
            "Date": date,
            "Team": team_name,
            "Action": action,
            "Player": player,
            "Position": position,
        })

df = pd.DataFrame(rows)
df

## Filter Activity by Type

`msg_type` options: `'FA'`, `'WAIVER'`, `'TRADED'`

In [ ]:
for msg_type in ['FA', 'WAIVER', 'TRADED']:
    acts = league.recent_activity(size=25, msg_type=msg_type)
    print(f"\n{msg_type} activities: {len(acts)}")
    for act in acts[:5]:
        for team, action, player, position in act.actions:
            team_name = team.team_name if hasattr(team, 'team_name') else str(team)
            print(f"  {team_name:25s} {action:15s} {player}")

## Transactions API

`league.transactions()` returns `Transaction` objects with richer data:
- `team`, `type`, `status`, `scoring_period`, `date`, `bid_amount`
- `items`: list of `TransactionItem` with `type` ('ADD'/'DROP') and `player`

In [ ]:
txns = league.transactions()
print(f"Current period transactions: {len(txns)}\n")

for txn in txns[:20]:
    team_name = txn.team.team_name if hasattr(txn.team, 'team_name') else str(txn.team)
    date = datetime.fromtimestamp(txn.date / 1000).strftime('%Y-%m-%d %H:%M') if txn.date else 'N/A'
    items_str = ", ".join(f"{i.type}: {i.player}" for i in txn.items)
    bid = f" (bid: ${txn.bid_amount})" if txn.bid_amount else ""
    print(f"  [{txn.type:12s}] {date}  {team_name:25s} → {items_str}{bid}")

## Transactions by Type

In [ ]:
for txn_type in ['FREEAGENT', 'WAIVER', 'TRADE_ACCEPT']:
    txns = league.transactions(types={txn_type})
    print(f"\n{txn_type}: {len(txns)} transactions")
    for txn in txns[:5]:
        team_name = txn.team.team_name if hasattr(txn.team, 'team_name') else str(txn.team)
        items_str = ", ".join(f"{i.type}: {i.player}" for i in txn.items)
        print(f"  {team_name:25s} → {items_str}")

## Transaction Object Inspection

In [ ]:
txns = league.transactions()
if txns:
    txn = txns[0]
    print(f"Transaction object attributes:")
    for attr in sorted(dir(txn)):
        if not attr.startswith('_'):
            val = getattr(txn, attr)
            if not callable(val):
                print(f"  {attr}: {val}")

    print(f"\nTransaction items:")
    for item in txn.items:
        print(f"  type: {item.type}, player: {item.player}")
        for attr in sorted(dir(item)):
            if not attr.startswith('_') and not callable(getattr(item, attr)):
                print(f"    {attr}: {getattr(item, attr)}")
else:
    print("No transactions in current period.")